In [109]:
import collections
import json
import pickle
import re

import pandas as pd

import importlib
import arkham_reload
importlib.reload(arkham_reload)
arkham_reload.reload_arkham_modules()


# Configuration
TABOO_FLAG = 'CURRENT'  # match combined.ipynb behaviour

DECKLIST_JSON_PICKLE = 'decklist_json.pickle'
CARD_JSON_PICKLE = 'card_json.pickle'
TABOO_FILE = 'taboo.json'

# Load previously-scraped decklists and cards from disk
with open(DECKLIST_JSON_PICKLE, 'rb') as f:
  decklist_json = pickle.loads(f.read())
with open(CARD_JSON_PICKLE, 'rb') as f:
  card_json = pickle.loads(f.read())

print(f"Loaded {len(decklist_json)} raw decklists")
print(f"Loaded {len(card_json)} cards")

Loaded 63270 raw decklists
Loaded 2063 cards


In [110]:
# Basic decklist cleaning: drop empty entries, joke decks, extreme copy violations

from arkham_popularity import clean_decklist_json

decklist_json, removed_decks = clean_decklist_json(
  decklist_json,
  card_json,
  TABOO_FILE,
)

for deck_id, reason, detail in removed_decks:
  label = detail or ''
  print(f"  removed {deck_id} ({reason}): {label}")

print(f"{len(decklist_json)} decklists after removing empty/joke/illegal copies")

  removed 25318 (empty): 
  removed 25293 (empty): 
  removed 25292 (empty): 
  removed 25291 (empty): 
  removed 25245 (empty): 
  removed 25243 (empty): 
  removed 25241 (empty): 
  removed 25240 (empty): 
  removed 25239 (empty): 
  removed 25237 (empty): 
  removed 25236 (empty): 
  removed 25235 (empty): 
  removed 25234 (empty): 
  removed 25232 (empty): 
  removed 25224 (empty): 
  removed 25223 (empty): 
  removed 25222 (empty): 
  removed 25221 (empty): 
  removed 25206 (empty): 
  removed 25204 (empty): 
  removed 25200 (empty): 
  removed 25199 (empty): 
  removed 25195 (empty): 
  removed 25194 (empty): 
  removed 25193 (empty): 
  removed 25190 (empty): 
  removed 25189 (empty): 
  removed 25188 (empty): 
  removed 25187 (empty): 
  removed 25179 (empty): 
  removed 25176 (empty): 
  removed 25170 (empty): 
  removed 25168 (empty): 
  removed 25129 (empty): 
  removed 25128 (empty): 
  removed 25115 (empty): 
  removed 25112 (empty): 
  removed 25111 (empty): 
  removed 25

In [111]:
# Load taboo data and helpers (adapted from combined.ipynb)

with open(TABOO_FILE, 'r', encoding='utf-8') as f:
  taboo_json = json.load(f)

# Find id of most recent taboo
MAX_TABOO = max([taboo_dict['id'] for taboo_dict in taboo_json])
print(f"Most recent taboo id: {MAX_TABOO}")

# Build lookup: taboo_id -> {card_code -> taboo_card}
taboo_dict = collections.defaultdict(dict)
for taboo in taboo_json:
  taboo_id = taboo['id']
  for taboo_card in json.loads(taboo['cards']):
    card_code = taboo_card['code']
    taboo_dict[taboo_id][card_code] = taboo_card

  if taboo_id == MAX_TABOO:
    # list of card_codes in most recent taboo that change xp or are forbidden
    tabooed_cards = [card['code'] for card in json.loads(taboo['cards'])
                     if 'xp' in card or card['text'] == 'Forbidden.']


def lookup_taboo(card_code: str, taboo_id: int | None):
  """Return taboo card entry for (card_code, taboo_id), or None if missing."""
  if taboo_id in taboo_dict:
    return taboo_dict[taboo_id].get(card_code, None)
  return None


def lookup_taboo_xp(card_code: str, taboo_id: int | None) -> int:
  """Return taboo xp modifier for (card_code, taboo_id), defaulting to 0."""
  taboo_card = lookup_taboo(card_code, taboo_id)
  if taboo_card is not None:
    return taboo_card.get('xp', 0)
  return 0

Most recent taboo id: 10


In [112]:
# Map card_id -> canonical_id (spec.md fingerprint + duplicate_of_code)
# and rewrite decklist slots / taboo keys to canonical_id.

from arkham_canonical import CanonicalMapper, parse_investigator_front_back

canonical_mapper = CanonicalMapper(card_json, chapter=1)
canonical_id_map = canonical_mapper.canonical_id_map
canonical_cycle = canonical_mapper.canonical_cycle


def to_canonical(card_id: str) -> str:
  """Return canonical_id for a card_id (identity if unmapped)."""
  return canonical_mapper.to_canonical(card_id)


def to_canonical_front(card_id: str) -> str:
  """Return canonical_front for an investigator front card_id."""
  return canonical_mapper.to_canonical_front(card_id)


def to_canonical_back(card_id: str) -> str:
  """Return canonical_back for an investigator back card_id."""
  return canonical_mapper.to_canonical_back(card_id)


def decklist_canonical_front_back(decklist: dict) -> tuple[str, str]:
  """Return (canonical_front, canonical_back) for a decklist."""
  return canonical_mapper.decklist_canonical_front_back(decklist)


def cycle_for_slot(card_id: str) -> int | None:
  """Return CanonicalCard.cycle for a slot code, or None if out-of-order or missing."""
  return canonical_mapper.cycle_for_slot(card_id)


def decklist_cycle(slots: dict, *, canonical_front: str | None = None) -> int | None:
  """Return Decklist.cycle: max cycle among player-chosen slot codes."""
  return canonical_mapper.decklist_cycle(slots, canonical_front=canonical_front)


def decklist_has_unknown_slots(slots: dict) -> bool:
  """Return True if any slot code is missing from card_json."""
  return canonical_mapper.decklist_has_unknown_slots(slots)


# Merge decklist card codes
for decklist in decklist_json.values():
  slots = decklist['slots']
  for card_code in list(slots.keys()):
    canonical_id = to_canonical(card_code)
    if canonical_id == card_code:
      continue

    num = slots.pop(card_code)
    slots[canonical_id] = slots.get(canonical_id, 0) + num


# Merge taboo card codes so taboo lookups stay consistent
for taboo_id in sorted(taboo_dict.keys()):
  taboo = taboo_dict[taboo_id]
  for card_code, taboo_card in list(taboo.items()):
    canonical_id = to_canonical(card_code)
    if canonical_id == card_code:
      continue
    taboo_dict[taboo_id][canonical_id] = taboo_card

# tabooed_cards is used for CURRENT-taboo xp checks on canonical slot codes
tabooed_cards = sorted({to_canonical(card_code) for card_code in tabooed_cards})

reprint_card_ids = sum(1 for card_id, canonical_id in canonical_id_map.items()
                       if card_id != canonical_id)
print(f"Mapped {len(card_json)} card_ids -> {len(set(canonical_id_map.values()))} canonical_ids")
print(f"Merged {reprint_card_ids} reprint card_ids in decklists and taboo data")

Mapped 2063 card_ids -> 1666 canonical_ids
Merged 147 reprint card_ids in decklists and taboo data


In [113]:
# Cycle stats (spec Pack Order; uses CanonicalCard.cycle from arkham_canonical)

cycle_card_count = collections.Counter()
for card_code, card in card_json.items():
  # Ignore investigator-specific signature cards
  if 'restrictions' in card and 'investigator' in card['restrictions']:
    continue
  # Ignore basic weaknesses
  if card.get('subtype_code') == 'basicweakness':
    continue
  # Count each canonical_id once
  if to_canonical(card_code) != card_code:
    continue

  cycle = cycle_for_slot(card_code)
  if cycle is None:
    continue
  cycle_card_count[cycle] += 1

cycle_card_cumulative: dict[int, int] = {}
so_far = 0
for cycle in sorted(cycle_card_count):
  so_far += cycle_card_count[cycle]
  cycle_card_cumulative[cycle] = so_far

deck_cycle_by_card_cycle: dict[int, dict[int, int]] = collections.defaultdict(
  lambda: collections.defaultdict(int)
)
unknown_slot_refs = 0
for decklist in decklist_json.values():
  slots = decklist.get('slots') or {}
  canon_front, _ = decklist_canonical_front_back(decklist)
  deck_cycle = decklist_cycle(slots, canonical_front=canon_front)
  if deck_cycle is None:
    continue

  for card_code, num in slots.items():
    if not canonical_mapper.is_known_card(card_code):
      unknown_slot_refs += num
      continue
    card_cycle = cycle_for_slot(card_code)
    if card_cycle is None:
      continue
    deck_cycle_by_card_cycle[deck_cycle][card_cycle] += num

cycle_pivot = pd.DataFrame.from_dict(
  {deck_cycle: dict(card_counts) for deck_cycle, card_counts in deck_cycle_by_card_cycle.items()},
  orient='index',
).fillna(0).astype(int)
cycle_pivot.index.name = 'Decklist.cycle'
cycle_pivot.columns.name = 'CanonicalCard.cycle'
cycle_pivot = cycle_pivot.sort_index().reindex(sorted(cycle_pivot.columns), axis=1)

cycle_pivot['All'] = cycle_pivot.sum(axis=1)
all_row = cycle_pivot.drop(columns='All').sum(axis=0)
all_row['All'] = int(cycle_pivot.drop(columns='All').values.sum())
cycle_pivot.loc['All'] = all_row

print("Cycle card counts:")
print(dict(sorted(cycle_card_count.items())))
print("Cumulative cards by cycle:")
print(cycle_card_cumulative)
print("Slot copies by Decklist.cycle x CanonicalCard.cycle (margins reproduce prior totals):")
display(cycle_pivot)
if unknown_slot_refs:
  print(f"Skipped {unknown_slot_refs} slot copies with unknown card codes")

Cycle card counts:
{1: 86, 2: 107, 3: 112, 4: 114, 5: 105, 6: 96, 7: 122, 8: 116, 9: 139, 10: 114, 11: 119, 12: 126, 13: 215}
Cumulative cards by cycle:
{1: 86, 2: 193, 3: 305, 4: 419, 5: 524, 6: 620, 7: 742, 8: 858, 9: 997, 10: 1111, 11: 1230, 12: 1356, 13: 1571}
Slot copies by Decklist.cycle x CanonicalCard.cycle (margins reproduce prior totals):


CanonicalCard.cycle,1,2,3,4,5,6,7,8,9,10,11,12,13,All
Decklist.cycle,,,,,,,,,,,,,,
1,79021,82,125,8,11,5,3,1,14,0,1,1,1,79273
2,146139,59570,361,57,33,37,9,10,5,0,1,0,0,206222
3,105505,42892,37639,620,59,35,2,4,10,11,0,2,0,186779
4,76958,33543,31584,27845,186,70,13,36,1,0,1,0,0,170237
5,102174,43237,39393,33467,46937,1065,26,64,8,0,2,0,7,266380
6,44054,20378,20094,17677,22480,18618,15,69,0,6,0,0,0,143391
7,67454,25933,18599,16824,15226,10108,43993,397,71,58,0,4,2,198669
8,31285,13191,11106,10234,12510,10269,11333,16574,61,7,7,43,2,116622
9,61130,21690,21143,15332,15950,12205,18378,9109,27272,169,13,31,1,202423


Skipped 1 slot copies with unknown card codes


In [114]:
# Normalize by row
cycle_pivot.div(cycle_pivot.sum(axis=1) / 2, axis=0)

CanonicalCard.cycle,1,2,3,4,5,6,7,8,9,10,11,12,13,All
Decklist.cycle,,,,,,,,,,,,,,
1,0.996821,0.001034,0.001577,0.000101,0.000139,0.000063,0.000038,0.000013,0.000177,0.000000,0.000013,0.000013,0.000013,1.0
2,0.708649,0.288863,0.001751,0.000276,0.000160,0.000179,0.000044,0.000048,0.000024,0.000000,0.000005,0.000000,0.000000,1.0
3,0.564865,0.229640,0.201516,0.003319,0.000316,0.000187,0.000011,0.000021,0.000054,0.000059,0.000000,0.000011,0.000000,1.0
4,0.452064,0.197037,0.185530,0.163566,0.001093,0.000411,0.000076,0.000211,0.000006,0.000000,0.000006,0.000000,0.000000,1.0
5,0.383565,0.162313,0.147883,0.125636,0.176203,0.003998,0.000098,0.000240,0.000030,0.000000,0.000008,0.000000,0.000026,1.0
6,0.307230,0.142115,0.140134,0.123278,0.156774,0.129841,0.000105,0.000481,0.000000,0.000042,0.000000,0.000000,0.000000,1.0
7,0.339530,0.130534,0.093618,0.084684,0.076640,0.050879,0.221439,0.001998,0.000357,0.000292,0.000000,0.000020,0.000010,1.0
8,0.268260,0.113109,0.095231,0.087754,0.107270,0.088054,0.097177,0.142117,0.000523,0.000060,0.000060,0.000369,0.000017,1.0
9,0.301991,0.107152,0.104450,0.075742,0.078795,0.060295,0.090790,0.045000,0.134728,0.000835,0.000064,0.000153,0.000005,1.0


In [115]:
# XP and investigator name helpers (adapted from combined.ipynb)

UPGRADE_PATTERN = re.compile(r"\d+\|\d+.*")


def parse_customizable(customizable_string: str):
  """Parse ArkhamDB customizable upgrade string.

  Returns (indices_list, xp_list) where each index is an upgrade index and
  each xp is its cost, filtered to upgrades with positive xp.
  """
  upgrade_list = customizable_string.split(',')
  upgrade_list = [u.split('|') for u in upgrade_list
                  if UPGRADE_PATTERN.match(u)]
  indices_list = [u[0] for u in upgrade_list if int(u[1]) > 0]
  xp_list = [int(u[1]) for u in upgrade_list if int(u[1]) > 0]
  return indices_list, xp_list


def calc_card_code_xp(card_code: str, taboo_id: int | None) -> int:
  card_dict = card_json[card_code]
  card_xp = card_dict.get('xp', 0)
  exceptional = card_dict.get('exceptional', False)

  # Apply taboo modifications if we are pinning to the latest taboo
  if TABOO_FLAG == 'CURRENT' and card_code in tabooed_cards:
    card_taboo = lookup_taboo(card_code, MAX_TABOO)
    if card_taboo is not None:
      if 'xp' in card_taboo:
        card_xp += card_taboo['xp']
      if 'exceptional' in card_taboo:
        exceptional = card_taboo['exceptional']

  if exceptional:
    card_xp *= 2

  return card_xp


def decklist_investigator_name(decklist: dict) -> str:
  """Return investigator name, treating parallel fronts/backs as distinct."""
  investigator_name = decklist['investigator_name']
  return investigator_name


def decklist_investigator_front_back(decklist: dict) -> tuple[str, str]:
  """Return raw investigator_front/back card_ids from a decklist."""
  return parse_investigator_front_back(decklist)


def decklist_xp(decklist: dict) -> int:
  """Return total XP of the decklist, including customizable upgrades."""
  xp = 0
  taboo_id = decklist.get('taboo_id')

  for card_code, num in decklist['slots'].items():
    card_dict = card_json[card_code]
    card_xp = calc_card_code_xp(card_code, taboo_id)

    if card_dict.setdefault('myriad', False):
      xp += card_xp
    else:
      xp += card_xp * num

  meta = decklist.get('meta', '')
  if meta:
    customizable_cards = json.loads(meta)
    for key, customizable_string in customizable_cards.items():
      if key.startswith('cus_'):
        _, xp_list = parse_customizable(customizable_string)
        xp += sum(xp_list)

  # Global XP adjustments
  if '08125' in decklist['slots']:
    xp = max(0, xp - 3)  # In the Thick of It

  if decklist_investigator_name(decklist) == 'Kymani Jones':
    xp = max(0, xp - 5)

  return xp

In [116]:
# Build decklist DataFrame via spec.md pipeline (D1–D4)
# Tolerates missing card_json entries until cards are re-scraped.
# Re-run this cell after editing arkham_*.py (reloads modules + rebuilds pop_engine).

from arkham_popularity import ArkhamPopularityEngine, prepared_decks_to_dataframe

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)
prepared_decks = pop_engine.prepare_all(decklist_json)
df_all_decks = prepared_decks_to_dataframe(prepared_decks)

print(df_all_decks.shape)
print(
  'Unknown slots:',
  int(df_all_decks['has_unknown_slots'].sum()),
  '| Chapter 2:',
  int(df_all_decks['has_chapter_2_cards'].sum()),
)
print(df_all_decks[[
  'id', 'investigator_name',
  'canonical_front', 'canonical_back',
  'cycle', 'xp_cost', 'is_ignore',
]].head())

(55809, 21)
Unknown slots: 1 | Chapter 2: 539
      id investigator_name canonical_front canonical_back  cycle  xp_cost  \
0  25345       Agnes Baker           01004          01004    5.0        0   
1  25344       Wendy Adams           01005          01005    4.0        0   
2  25343      Stella Clark           60501          60501    7.0        0   
3  25342       Agnes Baker           01004          01004    2.0        0   
4  25341      Roland Banks           01001          01001    1.0        5   

   is_ignore  
0      False  
1       True  
2      False  
3      False  
4      False  


In [117]:
# is_ignore was set by ArkhamPopularityEngine (D4: taboo_set, unknown slots,
# Chapter 2). Keep non-ignored decks for analysis.

df_decks = df_all_decks[~df_all_decks['is_ignore']].copy()
print(f"{len(df_decks)} non-ignored decks")

44571 non-ignored decks


In [118]:
# Spec Y1/Y2: user_weight and Cycle.weight

user_weight_by_id = pop_engine.assign_user_weights(prepared_decks)
df_decks['user_weight'] = df_decks['id'].map(user_weight_by_id)

cycle_weight_by_cycle = pop_engine.assign_cycle_weights(
  prepared_decks,
  user_weight_by_id,
)
df_decks['cycle_weight'] = df_decks['cycle'].map(cycle_weight_by_cycle)

df_decks['deck_weight'] = df_decks['user_weight'] * df_decks['cycle_weight']

print(f"Total user_weight: {df_decks['user_weight'].sum():.1f}")
print(f"Total deck_weight: {df_decks['deck_weight'].sum():.4f}")
print('Cycle weights:', {k: round(v, 6) for k, v in sorted(cycle_weight_by_cycle.items())})

# Sanity check (spec Y2): deck counts and sum_user_weight per cycle
cycle_sanity = (
  df_decks[df_decks['cycle'].notna()]
  .groupby('cycle', as_index=False)
  .agg(deck_count=('id', 'count'), sum_user_weight=('user_weight', 'sum'))
  .sort_values('cycle')
)
cycle_sanity['sum_user_weight'] = cycle_sanity['sum_user_weight'].round(1)
cycle_sanity['cycle_weight'] = (1.0 / cycle_sanity['sum_user_weight']).round(6)
print('\nDecks and sum_user_weight by cycle (non-ignored, cycle defined):')
display(cycle_sanity)

none_decks = df_decks[df_decks['cycle'].isna()]
if len(none_decks):
  print(
    f"cycle=None: {len(none_decks)} decks, "
    f"sum_user_weight={none_decks['user_weight'].sum():.1f}"
  )

Total user_weight: 21257.1
Total deck_weight: 10.0781
Cycle weights: {1: 0.000405, 2: 0.000405, 3: 0.000405, 4: 0.000405, 5: 0.000405, 6: 0.000442, 7: 0.000442, 8: 0.000442, 9: 0.000442, 10: 0.000534, 11: 0.000587, 12: 0.001147}

Decks and sum_user_weight by cycle (non-ignored, cycle defined):


,cycle,deck_count,sum_user_weight,cycle_weight
0,1.0,1975,1099.7,0.000909
1,2.0,4718,2151.4,0.000465
2,3.0,4357,1995.7,0.000501
3,4.0,3979,1844.1,0.000542
4,5.0,5804,2466.5,0.000405
5,6.0,2959,1341.8,0.000745
6,7.0,4633,2194.9,0.000456
7,8.0,2992,1450.8,0.000689
8,9.0,4485,2263.8,0.000442
9,10.0,3723,1872.1,0.000534


cycle=None: 2 decks, sum_user_weight=2.0


In [119]:
# deck_weight is the spec popularity weight (user_weight * Cycle.weight).
# Keep group_score as an alias for downstream cells until they are migrated.
df_decks['group_score'] = df_decks['deck_weight']
print(f"Total group_score (deck_weight): {df_decks['group_score'].sum():.4f}")

Total group_score (deck_weight): 10.0781


In [120]:
# Helper used throughout: adjust scores for uneven pack / cycle availability


def group_and_score(df: pd.DataFrame,
                    groupby_cols: list[str],
                    score_col: str,
                    pack_index_col: str):
  """Aggregate scores per group, normalized by pack availability.

  This is a direct analogue of the function in prepare_data.ipynb,
  specialized for Arkham where `pack_index_col` is publication cycle.
  """
  # For each (group, pack), total score
  df_group_pack = (
    df.groupby(groupby_cols + [pack_index_col])[score_col]
      .sum()
      .reset_index()
  )

  # For each pack, average score per group
  df_pack_map = (
    df_group_pack.groupby(pack_index_col)[score_col]
      .mean()
      .to_dict()
  )

  # Attach pack-wise average expectation
  df_group_pack['pack_avg'] = df_group_pack[pack_index_col].map(df_pack_map)

  # For each group, total score and total expected score
  df_group = (
    df_group_pack.groupby(groupby_cols)[[score_col, 'pack_avg']]
      .sum()
      .reset_index()
  )

  df_group['score'] = df_group[score_col] / df_group['pack_avg']
  return df_group.sort_values('score', ascending=False)

In [ ]:
# Investigator popularity (spec I1–I5): compare investigators within each inv_cycle.
# Pooling all inv_cycles in one table overweights late-cycle investigators because
# Decklist.cycle >= 12 decks are weighted heavily under g(C) = C.

inv_name_map = (
  df_decks[['canonical_front', 'canonical_back', 'investigator_name']]
  .drop_duplicates(['canonical_front', 'canonical_back'])
  .set_index(['canonical_front', 'canonical_back'])['investigator_name']
  .to_dict()
)

inv_pop_rows = pop_engine.investigator_popularity_by_cycle(prepared_decks)
investigator_popularity = pd.DataFrame(inv_pop_rows)
investigator_popularity['investigator_name'] = investigator_popularity.apply(
  lambda row: inv_name_map.get(
    (row['canonical_front'], row['canonical_back']),
    row['canonical_front'],
  ),
  axis=1,
)
investigator_popularity = investigator_popularity.rename(columns={
  'inv_cycle': 'Cycle',
  'popularity': 'Popularity',
})

display_cols = [
  'investigator_name', 'canonical_front', 'canonical_back',
  'Cycle', 'Popularity', # 'popularity_global',
  'investigator_weight', 'pool_weight',
]

pd.set_option('display.max_rows', None)
for inv_cycle in sorted(investigator_popularity['Cycle'].dropna().unique()):
  print(f'=== inv_cycle = {int(inv_cycle)} (pool: Decklist.cycle >= {int(inv_cycle)}) ===')
  subset = (
    investigator_popularity[investigator_popularity['Cycle'] == inv_cycle]
    .sort_values('Popularity', ascending=False)
  )
  display(subset[display_cols])

=== inv_cycle = 1 (pool: Decklist.cycle >= 1) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,popularity_global,investigator_weight,pool_weight
8,Daisy Walker,01002,01002,1,0.037139,0.037139,2.177675,58.63589
6,Roland Banks,01001,01001,1,0.036226,0.036226,2.124149,58.63589
3,Agnes Baker,01004,01004,1,0.026510,0.026510,1.554424,58.63589
9,Wendy Adams,01005,01005,1,0.017829,0.017829,1.045421,58.63589
4,"""Skids"" O'Toole",01003,01003,1,0.017605,0.017605,1.032295,58.63589
7,Daisy Walker,01002,90001,1,0.001073,0.001073,0.062910,58.63589
5,Roland Banks,01001,90024,1,0.000981,0.000981,0.057512,58.63589
2,"""Skids"" O'Toole",01003,90008,1,0.000580,0.000580,0.034001,58.63589
1,Agnes Baker,01004,90017,1,0.000289,0.000289,0.016962,58.63589
0,Wendy Adams,01005,90037,1,0.000269,0.000269,0.015779,58.63589


=== inv_cycle = 2 (pool: Decklist.cycle >= 2) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,popularity_global,investigator_weight,pool_weight
12,Zoey Samaras,02001,02001,2,0.043677,0.043393,2.541598,58.190319
11,Jenny Barnes,02003,02003,2,0.028197,0.028014,1.640797,58.190319
10,"""Ashcan"" Pete",02005,02005,2,0.021010,0.020858,1.222595,58.190319
15,Rex Murphy,02002,02002,2,0.019999,0.019857,1.163726,58.190319
18,Jim Culver,02004,02004,2,0.016266,0.016149,0.946505,58.190319
16,Zoey Samaras,02001,90059,2,0.000236,0.000235,0.013761,58.190319
13,Jim Culver,02004,90049,2,0.000164,0.000163,0.009554,58.190319
17,Jenny Barnes,02003,90084,2,0.000111,0.000110,0.006462,58.190319
14,"""Ashcan"" Pete",02005,90046,2,0.000092,0.000091,0.005342,58.190319


=== inv_cycle = 3 (pool: Decklist.cycle >= 3) ===


...

,investigator_name,canonical_front,canonical_back,Cycle,Popularity,popularity_global,investigator_weight,pool_weight
58,Charlie Kane,09018,09018,10,0.023115,0.012589,0.632369,27.357968
56,Kymani Jones,09008,09008,10,0.017655,0.008335,0.483016,27.357968
57,Vincent Lee,09004,09004,10,0.014594,0.007114,0.399269,27.357968
61,Carson Sinclair,09001,09001,10,0.013379,0.006498,0.366011,27.357968
60,Darrell Simmons,09015,09015,10,0.007670,0.003760,0.209846,27.357968
59,Amina Zidane,09011,09011,10,0.006320,0.003017,0.172904,27.357968


=== inv_cycle = 11 (pool: Decklist.cycle >= 11) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,popularity_global,investigator_weight,pool_weight
63,Alessandra Zorzi,10009,10009,11,0.025351,0.008042,0.471537,18.600477
64,Wilson Richards,10001,10001,11,0.015721,0.005078,0.292419,18.600477
65,Kate Winthrop,10004,10004,11,0.013302,0.004401,0.247423,18.600477
62,Hank Samson,10015,10015,11,0.010632,0.003555,0.197755,18.600477
66,Kōhaku Narukami,10012,10012,11,0.009339,0.003140,0.173719,18.600477


=== inv_cycle = 12 (pool: Decklist.cycle >= 12) ===


,investigator_name,canonical_front,canonical_back,Cycle,Popularity,popularity_global,investigator_weight,pool_weight
73,Michael McGlen,11011,11011,12,0.019359,0.002947,0.171570,8.862337
69,George Barnaby,11017,11017,12,0.019315,0.003195,0.171175,8.862337
68,Marion Tavares,11001,11001,12,0.015770,0.002935,0.139757,8.862337
70,Lucius Galloway,11004,11004,12,0.012308,0.002294,0.109074,8.862337
67,Gloria Goldberg,11014,11014,12,0.008919,0.006816,0.079042,8.862337
71,Agatha Crane,11008,11008,12,0.008495,0.001284,0.075284,8.862337
72,Agatha Crane,11007,11007,12,0.005798,0.000996,0.051384,8.862337


In [122]:
# Card popularity within a single investigator (spec P1–P5 + bias compensation)

DISPLAY_COLS = [
  'canonical_id', 'card_index', 'option_index', 'name',
  'cycle', 'xp', 'slot', 'P3', 'P4', 'Popularity',
]


def extract_investigator_cards(canonical_front: str,
                               canonical_back: str,
                               min_popularity: float = 0.0) -> pd.DataFrame:
  """Return spec P1–P5 popularity for a (canonical_front, canonical_back) tuple."""
  rows = pop_engine.popularity_for_investigator(
    prepared_decks,
    canonical_front,
    canonical_back,
  )
  df = pd.DataFrame(rows)
  if df.empty:
    return df
  df['cycle'] = df['canonical_id'].map(
    lambda cid: pop_engine.canonical_cards[cid].cycle
    if cid in pop_engine.canonical_cards else None
  )
  df = df[df['p5_popularity'] >= min_popularity]
  return df.rename(columns={
    'p3_opportunity_weight': 'P3',
    'p4_choice_weight': 'P4',
    'p5_popularity': 'Popularity',
  }).sort_values('Popularity', ascending=False)


def show_investigator_card_popularity(canonical_front: str,
                                      canonical_back: str,
                                      *,
                                      min_popularity: float = 0.0,
                                      head: int = 40) -> None:
  """Show separate 0 XP and 1+ XP popularity tables with CanonicalCard.cycle."""
  df = extract_investigator_cards(
    canonical_front, canonical_back, min_popularity=min_popularity,
  )
  inv_name = (
    df_decks.loc[
      (df_decks['canonical_front'] == canonical_front)
      & (df_decks['canonical_back'] == canonical_back),
      'investigator_name',
    ].iloc[0]
    if len(df_decks) else canonical_front
  )
  print(f'{inv_name} ({canonical_front} / {canonical_back})')

  zero_xp = df[df['xp'].fillna(0) == 0].copy()
  plus_xp = df[df['xp'].fillna(0) > 0].copy()

  # Format display columns: ints where applicable; blank (not 'None') for N/A text fields
  for table in [zero_xp, plus_xp]:
    if 'card_index' in table.columns:
      table['card_index'] = table['card_index'].fillna(0).astype(int)
    if 'cycle' in table.columns:
      table['cycle'] = table['cycle'].fillna(0).astype(int)
    if 'xp' in table.columns:
      table['xp'] = table['xp'].fillna(0).astype(int)
    if 'option_index' in table.columns:
      table['option_index'] = table['option_index'].fillna('')
    if 'slot' in table.columns:
      table['slot'] = table['slot'].fillna('')

  print(f'\n--- 0 XP cards ({len(zero_xp)} options) ---')
  display(zero_xp[DISPLAY_COLS].head(head))

  print(f'\n--- 1+ XP cards ({len(plus_xp)} options) ---')
  display(plus_xp[DISPLAY_COLS].head(head))


# Example: Daisy Walker (03002)
show_investigator_card_popularity('03002', '03002')

Minh Thi Phan (03002 / 03002)

--- 0 XP cards (444 options) ---


,canonical_id,card_index,option_index,name,cycle,xp,slot,P3,P4,Popularity
0,03010,1,,Analytical Mind,3,0,,44.952849,44.952849,1.000000
1,03011,1,,The King in Yellow,3,0,Hand,44.952849,44.952849,1.000000
2,01039,1,,Deduction,1,0,,64.696288,57.176833,0.883773
3,01039,2,,Deduction,1,0,,64.696288,54.765580,0.846503
4,01000,1,,Random Basic Weakness,1,0,,64.696288,50.210943,0.776102
5,01090,1,,Perception,1,0,,64.696288,46.961174,0.725871
6,01090,2,,Perception,1,0,,64.696288,41.989055,0.649018
7,02227,1,,Inquiring Mind,2,0,,51.329957,31.824009,0.619989
8,02227,2,,Inquiring Mind,2,0,,51.329957,28.904905,0.563120
9,03231,1,,Eureka!,3,0,,44.952849,25.199603,0.560579



--- 1+ XP cards (260 options) ---


,canonical_id,card_index,option_index,name,cycle,xp,slot,P3,P4,Popularity
22,60228,1,,Perception,7,2,,21.943864,7.601530,0.346408
24,60228,2,,Perception,7,2,,21.943864,6.731254,0.306749
25,05194,1,,Grisly Totem,5,3,Accessory,25.719842,7.735534,0.300761
28,11049,1,,Antikythera,12,5,Accessory,4.709987,1.297297,0.275435
38,01040,1,,Magnifying Glass,1,1,Hand,30.360754,7.209505,0.237461
40,10118,1,,Persistence,11,1,,10.693003,2.468230,0.230827
44,11089,1,,Ample Supplies,12,2,,4.709987,1.019109,0.216372
45,02150,1,,Deduction,2,2,,25.101923,5.309525,0.211519
54,02150,2,,Deduction,2,2,,25.101923,4.719858,0.188028
58,07196,1,,Unrelenting,8,1,,21.060469,3.786313,0.179783


In [123]:
# Number of assets in each slot (spec: weighted avg per asset slot type)

SLOT_USAGE_COLS = ['slot_type', 'weighted_avg', 'deck_count', 'total_weight']


def slot_usage_for_investigator(canonical_front: str,
                                canonical_back: str) -> pd.DataFrame:
  """Weighted average asset copies per asset slot type for one investigator tuple."""
  rows = pop_engine.slot_usage_for_investigator(
    prepared_decks, canonical_front, canonical_back
  )
  return pd.DataFrame(rows)


def show_slot_usage_for_investigator(canonical_front: str,
                                     canonical_back: str) -> None:
  """Display weighted average assets per asset slot type (Accessory, Ally, ...)."""
  df = slot_usage_for_investigator(canonical_front, canonical_back)
  inv_name = (
    df_decks.loc[
      (df_decks['canonical_front'] == canonical_front)
      & (df_decks['canonical_back'] == canonical_back),
      'investigator_name',
    ].iloc[0]
    if len(df_decks) else canonical_front
  )
  print(f'{inv_name} ({canonical_front} / {canonical_back})')
  if df.empty:
    print('No active decks for this investigator.')
    return
  display(
    df[SLOT_USAGE_COLS]
    .assign(weighted_avg=lambda t: t['weighted_avg'].round(3))
    .sort_values('weighted_avg', ascending=False)
    .reset_index(drop=True)
  )


# Example: Minh Thi Phan
show_slot_usage_for_investigator('01004', '01004')

Agnes Baker (01004 / 01004)


,slot_type,weighted_avg,deck_count,total_weight
0,Arcane,5.100,1530,68.619063
1,Hand,2.762,1530,68.619063
2,Accessory,2.504,1530,68.619063
3,Ally,2.455,1530,68.619063
4,Body,0.920,1530,68.619063
5,Mask,0.295,1530,68.619063
6,Tarot,0.100,1530,68.619063
7,Head,0.000,1530,68.619063


In [124]:
# Automatic decklist generation (spec: Automatic Decklist Generation)
# Re-run cell 7 first if you only edited arkham_*.py without re-running it.

from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

import pandas as pd


def show_generated_decklist(
    canonical_front: str,
    canonical_back: str,
    *,
    investigator_option=None,
) -> None:
  """Build and display a synthetic 0 XP deck from popularity + slot averages.

  For investigators with deck_options choices, pass investigator_option to pick
  a branch explicitly, e.g. investigator_option='guardian+survivor' for Charlie
  Kane or investigator_option={'deck_size': 40, 'faction': 'rogue'} for Mandy.
  When omitted, the most popular meta-backed choice is used.
  """
  result = pop_engine.generate_decklist(
    prepared_decks,
    canonical_front,
    canonical_back,
    investigator_option=investigator_option,
  )
  print(
    f'{result.investigator_name} ({canonical_front} / {canonical_back})'
  )
  if result.option_resolutions:
    choices = ', '.join(
      f'{resolution.kind}={resolution.choice}'
      for resolution in result.option_resolutions
    )
    print(f'Resolved options: {choices}')
  if result.skipped_reason:
    print(f'Skipped: {result.skipped_reason}')
    return

  print(
    f'Generated {result.deck_count} / {result.deck_size} player slots '
    f'({len(result.entries)} cards shown, weaknesses omitted)'
  )
  df = pd.DataFrame(result.entries)
  if df.empty:
    display(df)
    return
  for col in ('cycle', 'xp', 'count'):
    if col in df.columns:
      df[col] = df[col].fillna(0).astype(int)
  display(df[['canonical_id', 'count', 'name', 'cycle', 'slot', 'type_code']])


show_generated_decklist('04001', '04001')

Leo Anderson (04001 / 04001)
Resolved options: signature_select=04006, signature_select=04007
Generated 30 / 30 player slots (26 cards shown, weaknesses omitted)


,canonical_id,count,name,cycle,slot,type_code
0,01018,2,Beat Cop,1,Ally,asset
1,01021,2,Guard Dog,1,Ally,asset
2,01025,2,Vicious Blow,1,,skill
3,01088,2,Emergency Cache,1,,event
4,01091,2,Overpower,1,,skill
5,03158,2,Calling in Favors,3,,event
6,01016,1,.45 Automatic,1,Hand,asset
7,01020,1,Machete,1,Hand,asset
8,01048,1,Leo De Luca,1,Ally,asset
9,02184,1,Prepared for the Worst,2,,event


In [125]:
# Generation coverage by investigator
# Re-run cell 7 first if you only edited arkham_*.py without re-running it.

from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

import pandas as pd

coverage = pd.DataFrame(pop_engine.list_generatable_investigators(prepared_decks))
supported = coverage[coverage['supported']]
print(f'{len(supported)} / {len(coverage)} investigators supported for generation')
display(
  coverage.sort_values(['supported', 'name'], ascending=[False, True])[
    ['name', 'canonical_front', 'supported', 'skip_reason', 'training_decks']
  ]
)

94 / 100 investigators supported for generation


,name,canonical_front,supported,skip_reason,training_decks
14,"""Ashcan"" Pete",02005,True,NaN,1010
86,"""Ashcan"" Pete",90046,True,NaN,46
2,"""Skids"" O'Toole",01003,True,NaN,1062
7,"""Skids"" O'Toole",01503,True,NaN,0
82,"""Skids"" O'Toole",90008,True,NaN,145
60,Agatha Crane,11007,True,NaN,27
61,Agatha Crane,11008,True,NaN,31
3,Agnes Baker,01004,True,NaN,1531
8,Agnes Baker,01504,True,NaN,0
83,Agnes Baker,90017,True,NaN,77


In [86]:
from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

from pathlib import Path

GENERATED_DIR = Path('generated')
written = pop_engine.update_generated_decklist_csvs(
  prepared_decks, 'Fix asset slot target', GENERATED_DIR, diagnostics=True
)
print(f'Wrote {len(written)} files to {GENERATED_DIR.resolve()}/')
print('Example deck csv:', next(p.name for p in written if p.suffix == '.csv' and 'resolution' not in p.name))
print('Example resolution csv:', next((p.name for p in written if 'resolution' in p.name), 'none'))
print('Example version csv:', next((p.name for p in written if 'version' in p.name), 'none'))

Wrote 192 files to C:\Users\jgf11\Documents\GitHub\ahlcg2\generated/
Example deck csv: Roland Banks 01001.csv
Example resolution csv: Roland Banks 01001 resolution.csv
Example version csv: Roland Banks 01001 version.md


In [44]:
# Export generated-deck popularity CSVs (one file per supported investigator)
# Re-run cell 7 first if you only edited arkham_*.py without re-running it.

from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

from pathlib import Path

GENERATED_DIR = Path('generated')
written = pop_engine.export_generated_decklist_csvs(
    prepared_decks, GENERATED_DIR, diagnostics=True,
)
print(f'Wrote {len(written)} files to {GENERATED_DIR.resolve()}/')
print('Example deck csv:', next(p.name for p in written if p.suffix == '.csv' and 'resolution' not in p.name))
print('Example resolution csv:', next((p.name for p in written if 'resolution' in p.name), 'none'))

Wrote 69 files to C:\Users\jgf11\Documents\GitHub\ahlcg2\generated/
Example deck csv: Roland Banks 01001.csv
Example resolution csv: Mandy Thompson 06002 resolution.csv


In [45]:
# Bulk export above writes the default popularity/meta winner per investigator.
# For explicit deck_options branches, export a variant file instead, e.g.:
from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

charlie_variant = pop_engine.export_generated_decklist(
    prepared_decks,
    '09018',
    '09018',
    GENERATED_DIR,
    investigator_option='guardian+survivor',
    diagnostics=True,
)
print('Charlie variant files:', [p.name for p in charlie_variant])

Charlie variant files: ['Charlie Kane 09018 guardian+survivor.csv', 'Charlie Kane 09018 guardian+survivor resolution.csv']


In [48]:
# Bulk export above writes the default popularity/meta winner per investigator.
# For explicit deck_options branches, export a variant file instead, e.g.:
from arkham_popularity import ArkhamPopularityEngine

pop_engine = ArkhamPopularityEngine(card_json, canonical_mapper, taboo_json)

from pathlib import Path
GENERATED_DIR = Path('generated')
generated_deck = pop_engine.export_generated_decklist(
    prepared_decks,
    '08010',
    '08010',
    GENERATED_DIR,
    # investigator_option='guardian+survivor',
    diagnostics=True,
)
print('Generated files:', [p.name for p in generated_deck])

Generated files: ['Lily Chen 08010.csv', 'Lily Chen 08010 resolution.csv']
